<a href="https://colab.research.google.com/github/techasit239/DADS7202_PigPicture/blob/main/FreshCheck_Phase2_Exp1_SAM3_v1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# FreshCheck Phase 2 — Exp 1: SAM 3 + Text Prompt

**Step 1 · Experiment 1: SAM 3 zero-shot segmentation with text prompt "raw meat"**

> 🔗 **ต้องรัน Phase 2 Foundation ก่อน** — Exp 1 อ่าน GT masks + test CSV ที่ Foundation สร้าง

---

## Exp 1 ทำอะไร?

ใช้ **SAM 3** (Meta, Nov 2025) — foundation model สำหรับ promptable concept segmentation — ลองใส่ text prompt หลายแบบ เพื่อหาว่า prompt ไหน segment เนื้อหมูจากบรรจุภัณฑ์ได้ดีที่สุด

**ไม่ต้องเทรนอะไร** — SAM 3 เป็น zero-shot model ที่ใช้ได้ทันที

| ขั้น | งาน | เวลา |
|---|---|---|
| 1.0 | Setup + HuggingFace auth | 1 นาที |
| 1.1 | Install + load SAM 3 model | 3-5 นาที |
| 1.2 | Test ก่อนรันทั้ง dataset (sanity check) | 1 นาที |
| 1.3 | **ลอง 3 text prompts ดูว่าอันไหนดีสุด** | 5-10 นาที |
| 1.4 | รัน best prompt บนทั้ง 150 รูป | ~10-15 นาที |
| 1.5 | Evaluate IoU + Dice | 30 วินาที |
| 1.6 | Save predicted masks + results | 1 นาที |

**⏱ รวม ~20-30 นาที** (T4 GPU)

---

## ⚠️ ก่อนเริ่ม — ต้อง request access SAM 3

SAM 3 เป็น **gated model** ต้องขอ access ก่อน (อนุมัติประมาณ 1-2 ชั่วโมง):

1. ไปที่ https://huggingface.co/facebook/sam3
2. กดปุ่ม **"Request access"** กรอกข้อมูลตามฟอร์ม Meta
3. รอ email confirmation (มัก approve ภายในชั่วโมง)
4. ที่ HF Settings → Access Tokens → **Create new token** (read permission พอ)
5. เก็บ token ไว้ — จะใช้ login ใน cell 1.0.2

ถ้ายังไม่ได้ access → notebook นี้รันไม่ได้ ให้ทำ Exp 2 หรือ Exp 3 ก่อน


## 1.0 — Setup Environment + Authenticate

เปิด GPU (Runtime → Change runtime type → T4 GPU) ก่อนรัน


In [3]:
!pip install -q -U transformers accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 121.6 MB/s eta 0:00:00


In [2]:
# Test HF SAM3 Access
from huggingface_hub import login
from google.colab import userdata
login(token=userdata.get('HF_TOKEN'))

from transformers import Sam3Processor
processor = Sam3Processor.from_pretrained('facebook/sam3')
print('[OK] SAM 3 access verified — พร้อมรัน Phase 2 Exp 1')

OSError: You are trying to access a gated repo.
Make sure to have access to it at https://huggingface.co/facebook/sam3.
403 Client Error. (Request ID: Root=1-6a098218-3f9247e126a6d2445244921e;3cd062cd-1b0a-47a2-9f8a-648422f41fe4)

Cannot access gated repo for url https://huggingface.co/facebook/sam3/resolve/main/processor_config.json.
Your request to access model facebook/sam3 is awaiting a review from the repo authors.

In [ ]:
# 1.0.1 — Imports + paths
import os, sys, json, time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import torch

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'PyTorch: {torch.__version__}')
print(f'Device:  {device}')
if torch.cuda.is_available():
    print(f'GPU:     {torch.cuda.get_device_name(0)}')


In [ ]:
# 1.0.2 — Mount Drive + define paths
from google.colab import drive
drive.mount('/content/drive')

PROJECT_ROOT = '/content/drive/MyDrive/FreshCheck'
PHASE2_DIR     = f'{PROJECT_ROOT}/phase2'
THAI_TEST_CSV  = f'{PHASE2_DIR}/thai_test_set.csv'
GT_MASKS_DIR   = f'{PHASE2_DIR}/gt_masks'
CONFIGS_DIR    = f'{PROJECT_ROOT}/configs'

EXP1_DIR       = f'{PHASE2_DIR}/exp1_sam3'
EXP1_MASKS_DIR = f'{EXP1_DIR}/predicted_masks'
EXP1_RESULTS   = f'{EXP1_DIR}/results.json'
os.makedirs(EXP1_DIR, exist_ok=True)
os.makedirs(EXP1_MASKS_DIR, exist_ok=True)

# Validate Foundation outputs
assert os.path.exists(THAI_TEST_CSV), f'ไม่พบ {THAI_TEST_CSV} — รัน Phase 2 Foundation ก่อน'
assert os.path.exists(GT_MASKS_DIR),  f'ไม่พบ {GT_MASKS_DIR} — รัน Phase 2 Foundation ก่อน'

# Import shared utilities
sys.path.insert(0, CONFIGS_DIR)
from seg_metrics import compute_iou, compute_dice, evaluate_batch
from seg_viz     import show_seg_comparison, plot_iou_distribution

print(f'[OK] Phase 2 Foundation outputs verified')
print(f'     Test CSV: {THAI_TEST_CSV}')
print(f'     GT masks: {GT_MASKS_DIR}')


In [ ]:
# 1.0.3 — Authenticate with HuggingFace (for gated SAM 3)
# ⚠️ ต้องขอ access ที่ https://huggingface.co/facebook/sam3 ก่อน
# แล้วสร้าง token: HF Settings → Access Tokens → Create new token (read)
from huggingface_hub import login

# วิธีที่ 1 (แนะนำ): ใช้ Colab Secret
# - คลิก 🔑 ทางซ้าย → Add new secret → Name='HF_TOKEN' → ใส่ token → Notebook access ON
# - แล้วโหลดด้วย userdata.get
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN')
    print('[OK] Loaded HF_TOKEN from Colab Secrets')
except Exception:
    HF_TOKEN = None
    print('[!] ไม่มี Colab Secret "HF_TOKEN" — จะใช้วิธีที่ 2 (manual login)')

# วิธีที่ 2 (fallback): login ผ่าน prompt
if HF_TOKEN is None:
    print('Login เปิด prompt — paste HuggingFace token แล้ว Enter')
    login()  # interactive
else:
    login(token=HF_TOKEN)
    print('[OK] Authenticated with HuggingFace')


## 1.1 — Install + Load SAM 3

SAM 3 มี integration ใน HuggingFace `transformers` แล้ว แต่ต้อง upgrade เป็น version ใหม่


In [ ]:
# 1.1.1 — Install latest transformers (SAM 3 needs recent version)
!pip install -q -U transformers accelerate
import transformers
print(f'transformers: {transformers.__version__}')


In [ ]:
# 1.1.2 — Load SAM 3 model + processor
from transformers import Sam3Processor, Sam3Model

MODEL_ID = 'facebook/sam3'

print(f'Loading {MODEL_ID}...')
print('(ครั้งแรกจะดาวน์โหลด ~3.4 GB — ใช้เวลา 3-5 นาที)')

processor = Sam3Processor.from_pretrained(MODEL_ID)
model = Sam3Model.from_pretrained(MODEL_ID, dtype=torch.float16).to(device)
model.eval()

n_params = sum(p.numel() for p in model.parameters())
print(f'\n[OK] SAM 3 loaded')
print(f'     Parameters: {n_params/1e6:.0f}M')
print(f'     Precision: float16 (memory efficient on T4)')


## 1.2 — Sanity Check on 1 Image

ก่อนรันทั้ง 150 รูป → ทดสอบกับรูปเดียวเพื่อยืนยันว่า:
1. Model load + inference ทำงาน
2. Output format ตรงกับที่คาด
3. Mask ที่ได้สมเหตุสมผล (ไม่ใช่ noise สุ่ม)


In [ ]:
# 1.2.1 — Load test set + pick 1 sample image
df = pd.read_csv(THAI_TEST_CSV)
print(f'Test set: {len(df)} samples')

# Take 1 sample for sanity check
sample = df.iloc[0]
img_path = sample['image_path']
gt_path  = sample['mask_path']

img = Image.open(img_path).convert('RGB')
gt  = np.array(Image.open(gt_path).convert('L'))

print(f'\nSample: {sample["filename"]}')
print(f'  Class:  {sample["class"]}')
print(f'  Size:   {img.size}')
print(f'  GT foreground: {(gt > 127).mean() * 100:.1f}%')


In [ ]:
# 1.2.2 — Run SAM 3 on the sample with text prompt
def run_sam3(image, text_prompt, threshold=0.5):
    """
    Run SAM 3 with text prompt → returns union mask (HxW uint8 binary).
    SAM 3 may return multiple instance masks — union them for our task.
    """
    inputs = processor(images=image, text=text_prompt, return_tensors='pt').to(device, torch.float16)

    with torch.no_grad():
        outputs = model(**inputs)

    # Post-process: get masks at original image size
    # pred_masks shape: (batch, num_queries, H, W)
    pred_masks = processor.post_process_masks(
        outputs.pred_masks,
        original_sizes=[image.size[::-1]],  # PIL size (W,H) → (H,W)
        binarize=False,
    )[0]  # (num_queries, H_orig, W_orig)

    # Filter by presence score (which queries actually detected the concept)
    pred_logits = outputs.pred_logits[0]  # (num_queries,)
    confidence  = torch.sigmoid(pred_logits)
    keep = confidence > threshold

    if keep.sum() == 0:
        # No detection
        H, W = image.size[1], image.size[0]
        return np.zeros((H, W), dtype=np.uint8), 0

    selected = pred_masks[keep]  # (n_kept, H, W)
    # Take max across instances (union)
    union = (selected.sigmoid() > 0.5).any(dim=0).cpu().numpy().astype(np.uint8) * 255
    return union, int(keep.sum())


# Test with simplest prompt
test_prompt = 'raw meat'
t0 = time.time()
pred_mask, n_detections = run_sam3(img, test_prompt)
elapsed = time.time() - t0

print(f'Prompt: "{test_prompt}"')
print(f'  Inference time: {elapsed:.2f}s')
print(f'  Detections:     {n_detections} instance(s)')
print(f'  Predicted FG:   {(pred_mask > 0).mean() * 100:.1f}%')

# Compute metrics on this 1 sample
iou  = compute_iou(pred_mask, gt)
dice = compute_dice(pred_mask, gt)
print(f'  IoU:  {iou:.3f}')
print(f'  Dice: {dice:.3f}')


In [ ]:
# 1.2.3 — Visualize the sanity check result
fig = show_seg_comparison(img_path, gt, pred_mask,
                          title=f'SAM 3 sanity check · prompt="{test_prompt}" · IoU={iou:.3f}')
plt.show()


## 1.3 — Try 3 Text Prompts on Small Subset

อาจารย์แนะนำว่า "ลอง text prompt หลายๆ แบบ" — เลือก 3 prompts ที่น่าจะแตกต่างกัน แล้วลองรันบน subset เล็กๆ (~15 รูป × 3 = 45 inferences) ก่อน เพื่อตัดสินใจว่าจะใช้ prompt ไหนสำหรับ full eval

| Prompt | Strategy |
|---|---|
| `"raw meat"` | คำทั่วไป — น่าจะเข้าใจง่าย |
| `"pork tissue"` | เฉพาะเจาะจง — pork loin เป็นเนื้อหมู |
| `"red meat without packaging"` | บอก context ของ task |

ทำไม subset เล็ก: รัน full 150 × 3 prompts = 450 inferences = 30+ นาที — ไม่คุ้มถ้า prompt แรกไม่ work


In [ ]:
# 1.3.1 — Run 3 prompts on subset (stratified 5 รูป/class = 15 รูป)
PROMPTS = [
    'raw meat',
    'pork tissue',
    'red meat without packaging',
]

# Stratified sample: 5 รูปต่อ class
subset_df = df.groupby('class').sample(n=5, random_state=SEED).reset_index(drop=True)
print(f'Subset for prompt comparison: {len(subset_df)} images\n')

subset_results = {}
for prompt in PROMPTS:
    print(f'Prompt: "{prompt}"')
    ious, dices = [], []
    for _, row in subset_df.iterrows():
        img = Image.open(row['image_path']).convert('RGB')
        gt  = np.array(Image.open(row['mask_path']).convert('L'))
        pred, _ = run_sam3(img, prompt)
        ious.append(compute_iou(pred, gt))
        dices.append(compute_dice(pred, gt))
    subset_results[prompt] = {
        'iou_mean':  np.mean(ious),  'iou_std':  np.std(ious),
        'dice_mean': np.mean(dices), 'dice_std': np.std(dices),
    }
    print(f'  IoU:  {np.mean(ious):.3f} ± {np.std(ious):.3f}')
    print(f'  Dice: {np.mean(dices):.3f} ± {np.std(dices):.3f}\n')

# Pick best by mean IoU
BEST_PROMPT = max(subset_results, key=lambda p: subset_results[p]['iou_mean'])
print(f'★ Best prompt by mean IoU: "{BEST_PROMPT}"')
print(f'   (IoU = {subset_results[BEST_PROMPT]["iou_mean"]:.3f})')


## 1.4 — Full Evaluation on All 150 Images

รัน best prompt บน Thai test set ครบทุกรูป — บันทึก predicted mask แต่ละรูปไว้สำหรับ Phase 3 ใช้ต่อ


In [ ]:
# 1.4.1 — Full inference loop
from tqdm.auto import tqdm

print(f'Running SAM 3 with prompt "{BEST_PROMPT}" on {len(df)} images...')
print(f'(คาดว่าใช้เวลา ~{len(df) * 5 / 60:.0f}-{len(df) * 8 / 60:.0f} นาที)\n')

pred_masks_list = []
gt_masks_list = []
per_sample = []
inference_times = []

for idx, row in tqdm(df.iterrows(), total=len(df), desc='SAM 3'):
    img = Image.open(row['image_path']).convert('RGB')
    gt  = np.array(Image.open(row['mask_path']).convert('L'))

    t0 = time.time()
    pred, n_det = run_sam3(img, BEST_PROMPT)
    inference_times.append(time.time() - t0)

    # Save predicted mask
    mask_out = f'{EXP1_MASKS_DIR}/{Path(row["filename"]).stem}_pred.png'
    Image.fromarray(pred).save(mask_out)

    pred_masks_list.append(pred)
    gt_masks_list.append(gt)

    iou  = compute_iou(pred, gt)
    dice = compute_dice(pred, gt)
    per_sample.append({
        'filename': row['filename'],
        'class':    row['class'],
        'source':   row['source'],
        'iou':      iou,
        'dice':     dice,
        'n_detections': n_det,
    })

print(f'\n[OK] Done — avg inference time: {np.mean(inference_times):.2f}s/image')


In [ ]:
# 1.4.2 — Aggregate metrics
results = evaluate_batch(pred_masks_list, gt_masks_list)
per_sample_df = pd.DataFrame(per_sample)

print(f'OVERALL (n={results["n_samples"]}):')
print(f'  Mean IoU:  {results["mean_iou"]:.4f} ± {results["std_iou"]:.4f}')
print(f'  Mean Dice: {results["mean_dice"]:.4f} ± {results["std_dice"]:.4f}')

# Per class
print(f'\nPer class:')
for cls in per_sample_df['class'].unique():
    sub = per_sample_df[per_sample_df['class'] == cls]
    print(f'  {cls:12s} (n={len(sub):3d}): '
          f'IoU={sub["iou"].mean():.3f} ± {sub["iou"].std():.3f}, '
          f'Dice={sub["dice"].mean():.3f} ± {sub["dice"].std():.3f}')

# Per source (packaged vs unpackaged)
print(f'\nPer source:')
for src in per_sample_df['source'].unique():
    sub = per_sample_df[per_sample_df['source'] == src]
    print(f'  {src:12s} (n={len(sub):3d}): '
          f'IoU={sub["iou"].mean():.3f}, Dice={sub["dice"].mean():.3f}')


In [ ]:
# 1.4.3 — Plot IoU + Dice distribution
plot_iou_distribution(results, figsize=(12, 4))
plt.suptitle(f'Exp 1: SAM 3 + "{BEST_PROMPT}" — Distribution over 150 Thai test images', fontsize=12)
plt.tight_layout()
plt.savefig(f'{EXP1_DIR}/iou_dice_distribution.png', dpi=100, bbox_inches='tight')
plt.show()


In [ ]:
# 1.4.4 — Visualize: best, median, worst predictions
sorted_idx = np.argsort(per_sample_df['iou'].values)
indices_to_show = [
    ('Worst',  sorted_idx[0]),
    ('Median', sorted_idx[len(sorted_idx) // 2]),
    ('Best',   sorted_idx[-1]),
]

for label, idx in indices_to_show:
    row = df.iloc[idx]
    pred = pred_masks_list[idx]
    gt   = gt_masks_list[idx]
    iou = per_sample[idx]['iou']
    fig = show_seg_comparison(
        row['image_path'], gt, pred,
        title=f'{label} prediction · {row["filename"][:30]} · IoU={iou:.3f}'
    )
    plt.show()


## 1.5 — Save Results

บันทึก aggregate metrics + per-sample CSV ไว้ใน Drive ให้ Phase 3 อ่านเทียบกับ Exp 2 + 3


In [ ]:
# 1.5.1 — Save results JSON + per-sample CSV
final = {
    'experiment': 'Phase 2 Exp 1 — SAM 3 + text prompt (zero-shot)',
    'model_id': MODEL_ID,
    'best_prompt': BEST_PROMPT,
    'all_prompts_tried': subset_results,
    'config': {
        'n_test_samples': len(df),
        'precision': 'float16',
        'confidence_threshold': 0.5,
        'mask_threshold': 0.5,
        'seed': SEED,
    },
    'aggregate': {
        'mean_iou':  results['mean_iou'],
        'std_iou':   results['std_iou'],
        'mean_dice': results['mean_dice'],
        'std_dice':  results['std_dice'],
        'n_samples': results['n_samples'],
    },
    'per_class': {
        cls: {
            'n': int((per_sample_df['class'] == cls).sum()),
            'iou_mean':  float(per_sample_df[per_sample_df['class'] == cls]['iou'].mean()),
            'dice_mean': float(per_sample_df[per_sample_df['class'] == cls]['dice'].mean()),
        } for cls in per_sample_df['class'].unique()
    },
    'per_source': {
        src: {
            'n': int((per_sample_df['source'] == src).sum()),
            'iou_mean':  float(per_sample_df[per_sample_df['source'] == src]['iou'].mean()),
            'dice_mean': float(per_sample_df[per_sample_df['source'] == src]['dice'].mean()),
        } for src in per_sample_df['source'].unique()
    },
    'avg_inference_time_sec': float(np.mean(inference_times)),
    'output_dirs': {
        'predicted_masks': EXP1_MASKS_DIR,
    },
}

with open(EXP1_RESULTS, 'w') as f:
    json.dump(final, f, indent=2, ensure_ascii=False)

per_sample_df.to_csv(f'{EXP1_DIR}/per_sample_metrics.csv', index=False)

print(f'[OK] Saved:')
print(f'     {EXP1_RESULTS}')
print(f'     {EXP1_DIR}/per_sample_metrics.csv')
print(f'     {EXP1_MASKS_DIR}/ ({len(df)} predicted mask PNG files)')


In [ ]:
# 1.5.2 — Final summary
print('=' * 60)
print('PHASE 2 — Exp 1 (SAM 3 + text prompt) — DONE')
print('=' * 60)
print(f'\nBest prompt:  "{BEST_PROMPT}"')
print(f'Test samples: {results["n_samples"]}')
print(f'\nResults on Thai test set:')
print(f'  Mean IoU:  {results["mean_iou"]:.4f} ± {results["std_iou"]:.4f}')
print(f'  Mean Dice: {results["mean_dice"]:.4f} ± {results["std_dice"]:.4f}')
print(f'\nAverage inference time: {np.mean(inference_times):.2f}s/image')
print(f'\n[OK] Ready for comparison with Exp 2 + Exp 3')
